# Reading

We can open and close the door. Now let's try to get a reading.

Keep [captures/reading.txt](captures/reading.txt) open beside this notebook. Start at the comment about the user clicking Read. We will look at the earlier traffic later. Use your local Python kernel, pySerial, and the Pico's known COM/serial port.

The Pico's onboard LED is on when the simulated door is open and off when it is closed.


## 1. Start a fresh experiment

Disconnect the previous notebook (`pico.close()`), then unplug and reconnect your Pico to restore the default gain of 1 for comparison with the capture. Close any other application using its serial port. Recheck the port name if it changed.

Opening a new serial connection alone preserves the reader's settings and door position. The cell below the function definitions explicitly opens the door, so we know its position before our first attempt. To repeat the door experiment, run that cell again and continue in order.

In [ ]:
import serial
import time

PORT = "YOUR_PORT"  # Your Pico's COM/serial port.
pico = serial.Serial(PORT, baudrate=115200, timeout=1, write_timeout=1)
time.sleep(1)
pico.reset_input_buffer()


### Reuse the functions from the door practical

Here are `open_door()`, `close_door()` and `request_state()` from the previous notebook. Run these definitions to use them with this notebook's `pico` connection, or substitute your own implementations. Each function returns the reply bytes.


In [ ]:
def open_door():
    request = bytes.fromhex("A7 03 06 00 00 00 00 FF")
    pico.write(request)
    reply = pico.read(8)
    assert len(reply) == 8, "Incomplete Open reply."
    return reply


def close_door():
    request = bytes.fromhex("A7 03 07 00 00 00 00 FF")
    pico.write(request)
    reply = pico.read(8)
    assert len(reply) == 8, "Incomplete Close reply."
    return reply


def request_state():
    request = bytes.fromhex("A7 03 02 00 00 00 00 FF")
    pico.write(request)
    reply = pico.read(12)
    assert len(reply) == 12, "Incomplete state reply."
    return reply


In [ ]:
open_reply = open_door()
print("Open reply:", open_reply.hex(" "))

## 2. Reproduce the reading request

In `reading.txt`, find the comment about the user clicking Read. Look at the `>` line directly below it. Copy that request into the cell below.

The `<` line contains 198 bytes, much longer than the door replies. We will ask for that many bytes. With a one-second timeout, `read(198)` returns when it has 198 bytes or when the time runs out. If fewer arrive, it returns those bytes; it does not invent the missing ones.

**Predict:** do you expect the same kind of reply as the text file? Send the message and inspect what actually arrives.

In [ ]:
read_request = bytes.fromhex("")  # Copy the > line from reading.txt.
assert len(read_request) == 8, "Copy the eight-byte request before sending."
pico.write(read_request)
first_reply = pico.read(198)
print("Bytes received:", len(first_reply))
print("Reply:", first_reply.hex(" "))

## 3. Compare your reply with the example

Do not change the request yet. Compare your outgoing bytes with the text file again.

- Did we send the same message?
- How many reply bytes arrived? Does waiting for 198 mean the device must send 198?
- Does the reply still start and end like the messages we have seen?
- Does it repeat the command we sent?

Copy just the **first four bytes** of the successful `<` line into the cell below. Compare them with the first four bytes of your reply.

In [ ]:
successful_header = bytes.fromhex("")  # First four bytes of the < line.
assert len(successful_header) == 4, "Copy four bytes for this comparison."
print("Index / transcript / our reply")
for index in range(min(4, len(first_reply))):
    print(index, successful_header[index], first_reply[index])

**Your turn:** which byte differs? Previously, this byte stayed the same in the replies we examined. Could it describe the outcome of a request?

A short reply is not automatically a broken connection. Here we received a structured answer associated with our request. Before guessing what the differing value means, let's investigate what happened before the successful reading.

If you received no bytes, stop and check the connection. If you already received a long result, run the Open cell again and retry the read: another cell may have closed the door.

## 4. Look further back in the conversation

In [captures/reading.txt](captures/reading.txt), look above the Read action. Find `# Action: user clicks Close before reading`.

Compare its outgoing message with the commands from the door practical. Which operation do you recognize? Did we perform it before our read attempt?

Compare the state-query replies immediately before and after Close. Which byte changes? What did that byte tell us in the previous practical? Write down your explanation for why our reply differs from the captured reading.

## 5. Test the door hypothesis

First query your reader with `request_state()`. Save the reply, then use the familiar `close_door()` function and query again. We will change only the door position.

In [ ]:
state_before = request_state()
print("Before:", state_before.hex(" "))

In [ ]:
close_reply = close_door()
print("Close reply:", close_reply.hex(" "))
state_after = request_state()
print("Index / before / after")
for index in range(len(state_before)):
    print(index, state_before[index], state_after[index])

## 6. Retry the same reading

Keep `read_request` unchanged. Only the door position has changed.

**Predict:** will we get the same short reply, or a longer one now?

In [ ]:
pico.write(read_request)
reading_reply = pico.read(198)
print("Bytes received:", len(reading_reply))
print("First four bytes:", reading_reply[:4].hex(" "))
print("First reply:", first_reply.hex(" "))
print("New reply:", reading_reply.hex(" "))

**Your turn:** did closing the door change the outcome? Does the new header match the successful transcript? Did the command byte change between attempts, or only the reply?

Compare lengths and headers first. With the same settings, repeated measurements return identical values.

In [ ]:
print("Same header as the successful example?", reading_reply[:4] == successful_header)
print("Same response length as the example?", len(reading_reply) == 198)

## 7. Connect the outcome to the door position

We sent the same read request with the door open and closed. Compare your observations:

- Which state byte changed after Close?
- Which status byte and reply length accompanied each position?
- Did closing the door make the unchanged read request succeed?

These observations support the explanation that the reader disallows reading while its door is open. Copying the final request alone can fail when the device is in a different state.

## 8. Reopen the door and test again

A successful read might have changed something else. Reopen the door and send the same read request once more. **Predict:** will the short reply return?

Then close the door and read again. Keep each request together with its receive step. Save the original successful `reading_reply` for the decoding exercise.

In [ ]:
open_door()
pico.write(read_request)
open_again_reply = pico.read(198)
print("Read while open again:", open_again_reply.hex(" "))

close_door()
pico.write(read_request)
closed_again_reply = pico.read(198)
print("Read after closing again:", len(closed_again_reply), "bytes")

## 9. Construct a command: set gain

So far, we have copied complete commands from a transcript. Now we will **build a command from a Python number**. This lets us choose a gain without writing a new hex string each time.

We will put together three pieces: a fixed beginning, one byte for the gain, and a fixed ending. Keep your existing `pico` connection open.

### 1. See which part changes

In [captures/gain.txt](captures/gain.txt), these are the requests sent when the user clicks **Set gain**:

```text
                beginning       gain   ending
Set gain to 1:  A7 03 31 00 00 | 01 | 00 FF
Set gain to 2:  A7 03 31 00 00 | 02 | 00 FF
```

Only one byte changes: `01` becomes `02`. We will put our chosen gain there and keep all the other bytes as they appear in the capture. The command byte `31` selects Set gain.

### 2. Pack a number into one byte

`struct.pack("B", gain)` creates one byte containing our gain. `B` means a one-byte whole number from 0 to 255.

Run this cell with `gain = 2`. It prints `02`. For now, we are only creating a byte on the computer.

In [ ]:
import struct

gain = 2
gain_byte = struct.pack("B", gain)
print(gain_byte.hex(" "))

### 3. Join the three pieces

For `bytes`, `+` joins pieces together. The cell below puts our gain between the fixed beginning and ending.

With gain 2, the result should be `a7 03 31 00 00 02 00 ff`, matching the transcript. Creating `request` still does not send it.

In [ ]:
prefix = bytes.fromhex("A7 03 31 00 00")
suffix = bytes.fromhex("00 FF")
request = prefix + gain_byte + suffix

print(request.hex(" "))
print("Number of bytes:", len(request))
assert len(request) == 8

**Try gain 3:** change only `gain = 2` to `gain = 3` in step 2. Run that cell again, then run step 3 again.

The result should be `a7 03 31 00 00 03 00 ff`. You have constructed a new command from a number; you did not need to copy a gain-3 message from a capture.

### 4. Send the command you built

Run the cell below. `pico.write(request)` sends the bytes we just constructed. As before, we read the reply before sending anything else.

The reply repeats the gain byte. The check is supplied: its beginning is `A7 83 31 00 00`, followed by our gain byte and the same ending.

In [ ]:
pico.write(request)
reply = pico.read(8)
print("Reply:", reply.hex(" "))

expected_reply = bytes.fromhex("A7 83 31 00 00") + gain_byte + suffix
assert reply == expected_reply, "Unexpected gain reply: check the received bytes."

### 5. Put the construction in a function

We want to write `set_gain(2)` or `set_gain(3)` without rerunning several cells.

The function below already packs the gain into one byte, sends the command, and checks the reply. **Replace `None` with `prefix + gain_byte + suffix`**, the expression from step 3. That is the only line to complete.

The first check keeps the gain within the application's allowed range of whole numbers from 1 to 16. The function creates its own three pieces each time, using the number passed to it.

In [ ]:
def set_gain(gain):
    if type(gain) is not int or not 1 <= gain <= 16:
        raise ValueError("Gain must be a whole number from 1 to 16.")
    prefix = bytes.fromhex("A7 03 31 00 00")
    gain_byte = struct.pack("B", gain)
    suffix = bytes.fromhex("00 FF")
    request = None  # Replace None with prefix + gain_byte + suffix.
    assert request is not None, "Complete the request line above first."

    pico.write(request)
    reply = pico.read(8)
    expected_reply = bytes.fromhex("A7 83 31 00 00") + gain_byte + suffix
    assert reply == expected_reply, "Unexpected gain acknowledgement."
    return reply

After completing the function, run this cell. The first call sets gain 2; the second restores the default gain of 1 for the rest of the workshop.

Both calls use the same code. Only the number used to construct the command changes.

In [ ]:
print("Gain 2 reply:", set_gain(2).hex(" "))
print("Gain 1 reply:", set_gain(1).hex(" "))

You now have a function that **constructs and sends a command from a value**.

The main task is complete. You can explore the reading bytes in the optional section below, or jump to **section 11** to disconnect. The saved `reading_reply` is still the original gain-1 reading.

## 10. Explore the readings (optional)

We now have the reading reply, but printing a long string of hex does not yet give us useful measurements. **You can complete the loop below to try turning those bytes into numbers.**

The conversion format is supplied below. Focus on extracting the measurement bytes and checking the resulting numbers. This section is optional; it is fine to leave it unfinished when we move on.

Start with `reading_reply`, the successful response saved in section 6. Look at its beginning and end alongside the transcript. We have already compared its first four bytes; its final two bytes also resemble the ends of earlier replies.

Try this hypothesis: the first four and final two bytes surround the measurement data. Removing them from a 198-byte reply leaves 192 bytes. The application displays a 96-well plate: could those bytes represent 96 values of two bytes each? This is a format to test, not a layout we have proved.

### Convert each pair into a number

Use the supplied format `>H` to turn each pair of measurement bytes into a whole number:

```python
value, = struct.unpack(">H", pair)
```

Complete the loop below by adding that line and appending `value` to `readings`. The remaining questions are where the measurement data starts, how many values it contains, and how those values relate to the displayed plate.


### Check against the vendor display

Look at the vendor application values at the bottom of `reading.txt`. Copy a few corresponding byte pairs into a small `bytes.fromhex()` example and decode each pair with `>H`. Do the results match the displayed values?

Use bytes and displayed values from the same reading for an exact comparison. There is no need to load the file in Python. Check several values rather than relying on one match.


In [ ]:
import struct

payload = reading_reply[4:-2]  # Try treating the middle bytes as measurement data.
print("Candidate data bytes:", len(payload))
print("Possible two-byte values:", len(payload) // 2)

readings = []
for offset in range(0, len(payload), 2):
    pair = payload[offset:offset + 2]
    # Convert pair into a number, then append it to readings.
    # value, = struct.unpack(">H", pair)
    # readings.append(value)

print("Decoded values:", len(readings))
print("First twelve:", readings[:12])

### If you have time, investigate further

If you were able to complete the data extraction and get plausible numbers, you now have a flat list of 96 values. A 96-well plate is two-dimensional: **8 rows (A–H) and 12 columns (1–12)**. How could you turn `readings` into that shape?

Try building a list of rows so that `plate[0][0]` represents A1, `plate[0][11]` represents A12, and `plate[1][0]` represents B1.

- If the values arrive row by row, which slice of `readings` would make row A? What about row B?
- Can you complete the loop below to build all eight rows?
- Compare row A and H12 with the displayed values in the text file. Does that support your assumed ordering?

The shape alone does not tell us whether the device sends values row by row or column by column. Keep that question in mind as you explore. It is fine to leave this unfinished when we continue with the next clues.

In [ ]:
plate = []
for row_index in range(8):
    # Calculate where this row starts in readings.
    # Take the next 12 values and append that row to plate.
    # start = ...
    # row = readings[...:...]
    # plate.append(row)
    pass

for label, row in zip("ABCDEFGH", plate):
    print(label, row)

## 11. Disconnect

What was missing from our original attempt? What evidence helped distinguish a rejected request from an absent reply? Why would rerunning only the first read cell now give a different result than it did while the door was open?

We have obtained a reading and started exploring its contents. Keep any decoder experiments and unanswered questions for when we explore the next clues.

In [ ]:
pico.close()
print("Serial connection closed.")